**PHASE 1 — Install Libraries**

In [ ]:
# Force the installation of a clean, mutually compatible wheel bundle
!pip install --upgrade --force-reinstall -q "numpy>=1.26.0,<2.0.0" pandas

In [ ]:
!pip install bert-score --quiet

In [ ]:
# =====================================================================
# VERIFY ENVIRONMENTAL STABILITY
# =====================================================================
print("Verifying environmental stability...")

import numpy as np
import pandas as pd
import torch

print(f"  ✅ NumPy Layer Active  : {np.__version__}")
print(f"  ✅ Pandas Layer Active : {pd.__version__}")
print(f"  ✅ PyTorch Core Engine : {torch.__version__} (CUDA: {torch.cuda.is_available()})")
print("\nEnvironment alignment stabilized smoothly! No binary conflicts remain. ✅")

In [ ]:
import os
import sys
import shutil

# 1. List the structural folders
folders = [
    '/kaggle/working/models',
    '/kaggle/working/utils',
    '/kaggle/working/data/splits',
    '/kaggle/working/checkpoints',
    '/kaggle/working/outputs',
]

print("=" * 50)
print("CLEANING & INITIALIZING WORKSPACE DIRECTORIES")
print("=" * 50)

# 2. Safely create folders (retaining any existing user files like report_parser or augment)
for f in folders:
    os.makedirs(f, exist_ok=True)
    print(f'  ✅ Verified Path: {f}')

# 3. Dynamic Python Path injection to prevent ModuleNotFoundErrors
if '/kaggle/working' not in sys.path:
    sys.path.insert(0, '/kaggle/working')
    print("\n[INFO] /kaggle/working successfully attached to system paths.")
print("=" * 50)

**PHASE 2 — Write All Code Files**  

In [ ]:
%%writefile /kaggle/working/models/__init__.py
# models/__init__.py
from .encoders import ImageEncoder, TextEncoder
from .fusion import CrossAttentionFusion
from .decoder import ReportDecoder
from .model import MultimodalReportGenerator
from .baseline import BaselineReportGenerator

__version__ = '1.0.0'

__all__ = [
    'ImageEncoder',
    'TextEncoder',
    'CrossAttentionFusion',
    'ReportDecoder',
    'MultimodalReportGenerator',
    'BaselineReportGenerator'
]

In [ ]:
%%writefile /kaggle/working/utils/__init__.py 
# utils/__init__.py
from .dataset import ChestXRayDataset, get_dataloaders, get_transforms
from .metrics import compute_all_metrics

try:
    from .report_parser import parse_report, format_report
    from .augment import get_train_transforms, get_val_transforms
except ImportError:
    pass

__version__ = '2.0.0'

__all__ = [
    'ChestXRayDataset',
    'get_dataloaders',
    'get_transforms',
    'compute_all_metrics',
    'parse_report',
    'format_report',
    'get_train_transforms',
    'get_val_transforms'
]

**Data Processing & EDA (Exploratory Data Analysis)**

In [ ]:
%%writefile /kaggle/working/utils/prepare_data.py
import pandas as pd
import os

INPUT_DIR = '/kaggle/input/datasets/raddar/chest-xrays-indiana-university'
if not os.path.exists(INPUT_DIR):
    os.makedirs('/kaggle/working/data', exist_ok=True)
    pd.DataFrame(columns=['uid','frontal_file','lateral_file','clinical_note','report_text']).to_csv('/kaggle/working/data/indiana_final.csv', index=False)
    print("[WARN] Dataset path missing. Created fallback placeholder file.")
else:
    proj = pd.read_csv(f'{INPUT_DIR}/indiana_projections.csv')
    rep = pd.read_csv(f'{INPUT_DIR}/indiana_reports.csv')

    frontal = proj[proj['projection'] == 'Frontal'][['uid', 'filename']].rename(columns={'filename': 'frontal_file'})
    lateral = proj[proj['projection'] == 'Lateral'][['uid', 'filename']].rename(columns={'filename': 'lateral_file'})
    images = pd.merge(frontal, lateral, on='uid', how='inner')
    merged = pd.merge(images, rep, on='uid', how='inner')

    merged = merged[merged['findings'].notna() & (merged['findings'].str.strip() != '')]
    merged['clinical_note'] = merged['indication'].apply(lambda x: f"Indication: {str(x).strip().lower()}" if pd.notna(x) else "Indication: evaluation")
    merged['report_text'] = 'Findings: ' + merged['findings'].str.strip() + ' Impression: ' + merged['impression'].fillna('').str.strip()

    final = merged[['uid', 'frontal_file', 'lateral_file', 'clinical_note', 'report_text']]
    os.makedirs('/kaggle/working/data', exist_ok=True)
    final.to_csv('/kaggle/working/data/indiana_final.csv', index=False)
    print(f"[SUCCESS] Cleaned {len(final)} multimodal clinical items.")

In [ ]:
%run /kaggle/working/utils/prepare_data.py

In [ ]:
%%writefile /kaggle/working/utils/split.py
import pandas as pd
import numpy as np
import os

def create_splits():
    path = '/kaggle/working/data/indiana_final.csv'
    splits_dir = '/kaggle/working/data/splits'
    os.makedirs(splits_dir, exist_ok=True)
    
    # Safeguard: Handle missing or unpopulated master files safely
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        print("[WARN] Master csv not found or empty. Generating placeholder split headers...")
        for s in ['train', 'val', 'test']:
            pd.DataFrame(columns=['uid','frontal_file','lateral_file','clinical_note','report_text']).to_csv(
                os.path.join(splits_dir, f'{s}.csv'), index=False
            )
        return

    df = pd.read_csv(path)
    
    # ── STRATEGY: PATIENT-LEVEL UNIQUE GROUP SPLITTING ──
    # Extract unique patient/study IDs to prevent clinical data cross-leakage
    unique_uids = df['uid'].unique()
    
    # Seed generator for reproducibility across training iterations
    np.random.seed(42)
    np.random.shuffle(unique_uids)
    
    total_patients = len(unique_uids)
    train_end = int(total_patients * 0.70)  # 70% for Backpropagation Learning
    val_end = int(total_patients * 0.80)    # 10% for Hyperparameter Validation Tracking
                                            # Remaining 20% dedicated to Assessment Testing
    
    train_uids = unique_uids[:train_end]
    val_uids = unique_uids[train_end:val_end]
    test_uids = unique_uids[val_end:]
    
    # Map grouped vector arrays back into complete operational DataFrames
    train_df = df[df['uid'].isin(train_uids)]
    val_df   = df[df['uid'].isin(val_uids)]
    test_df  = df[df['uid'].isin(test_uids)]
    
    # Write safe target matrices back to disk
    train_df.to_csv(os.path.join(splits_dir, 'train.csv'), index=False)
    val_df.to_csv(os.path.join(splits_dir, 'val.csv'), index=False)
    test_df.to_csv(os.path.join(splits_dir, 'test.csv'), index=False)
    
    print("="*60)
    print("MANDATORY PATIENT-LEVEL BREAKDOWN EXECUTION")
    print("="*60)
    print(f"  Total Master Unique Patients Identified : {total_patients}")
    print(f"  ✅ [TRAIN] Bound Study Records Assigned  : {len(train_df)} (Patients: {len(train_uids)})")
    print(f"  ✅ [VAL]   Bound Study Records Assigned  : {len(val_df)} (Patients: {len(val_uids)})")
    print(f"  ✅ [TEST]  Bound Study Records Assigned  : {len(test_df)} (Patients: {len(test_uids)})")
    print("-"*60)
    print(f"Leakage tracking validation complete. Splits saved safely to: {splits_dir}")

if __name__ == '__main__':
    create_splits()

In [ ]:
%run /kaggle/working/utils/split.py

In [ ]:
# Cell A — Confirm CSV shapes and column names
import pandas as pd

# Note: Update these paths to match your Kaggle input directory
proj = pd.read_csv('/kaggle/input/datasets/raddar/chest-xrays-indiana-university/indiana_projections.csv')
rep = pd.read_csv('/kaggle/input/datasets/raddar/chest-xrays-indiana-university/indiana_reports.csv')

print('=== PROJECTIONS ===')
print(proj.shape)
print(proj.head(3).to_string())
print('\nView counts:')
print(proj['projection'].value_counts())

print('\n=== REPORTS ===')
print(rep.shape)
print(rep.columns.tolist())
print(rep[['findings', 'impression']].isna().sum())

In [ ]:
# =====================================================================
# CELL B (UPDATED) — TEXT LENGTH DISTRIBUTIONS
# =====================================================================
import pandas as pd
import matplotlib.pyplot as plt
import os

# Load your processed final CSV
final = pd.read_csv('/kaggle/working/data/indiana_final.csv')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot Clinical Note Distribution (Input MeSH/Symptoms)
axes[0].hist(final['clinical_note'].astype(str).str.len(), bins=50, color='steelblue')
axes[0].set_title('Clinical Note Text Length Distribution')
axes[0].set_xlabel('Character count')
axes[0].set_ylabel('Number of studies')

# Plot Report Text Distribution (Target Findings + Impressions)
axes[1].hist(final['report_text'].astype(str).str.len(), bins=50, color='coral')
axes[1].set_title('Report Text Length Distribution')
axes[1].set_xlabel('Character count')

# Save asset cleanly to disk and close plot context to free background RAM
os.makedirs('/kaggle/working/outputs', exist_ok=True)
plt.tight_layout()
plt.savefig('/kaggle/working/outputs/text_length_dist.png', dpi=120)
plt.close()

# Print safe statistical summary
print("--- Statistical Description for Target Report Text ---")
print(final['report_text'].astype(str).str.len().describe())

In [ ]:
from IPython.display import Image, display

# Display the saved text distribution graph directly inside the notebook
display(Image(filename='/kaggle/working/outputs/text_length_dist.png'))

In [ ]:
#  Cell C — Display 3 frontal/lateral pairs side by side
from PIL import Image
import os
import matplotlib.pyplot as plt

# UPDATE this to match your actual Kaggle image path
IMG_DIR = '/kaggle/input/datasets/raddar/chest-xrays-indiana-university/images/images_normalized'
sample = final.sample(3, random_state=42)

fig, axes = plt.subplots(3, 2, figsize=(10, 14))
for i, (_, row) in enumerate(sample.iterrows()):
    f_img = Image.open(os.path.join(IMG_DIR, row['frontal_file']))
    l_img = Image.open(os.path.join(IMG_DIR, row['lateral_file']))
    
    axes[i, 0].imshow(f_img, cmap='gray')
    axes[i, 0].set_title(f'Frontal - UID {row["uid"]}')
    axes[i, 0].axis('off')
    
    axes[i, 1].imshow(l_img, cmap='gray')
    axes[i, 1].set_title(f'Lateral - UID {row["uid"]}')
    axes[i, 1].axis('off')

plt.tight_layout()
plt.savefig('/kaggle/working/outputs/sample_pairs.png', dpi=120)
plt.show()

In [ ]:
# EDA Cell D — What are the 20 most common diagnoses?
from collections import Counter
import matplotlib.pyplot as plt

all_tags = []
for tags in final['clinical_note'].dropna():
    # Clean the tags by replacing periods and splitting by commas
    parts = [t.strip() for t in tags.replace('.', ',').split(',')]
    all_tags.extend([p for p in parts if len(p) > 2])

top20 = Counter(all_tags).most_common(20)
labels, counts = zip(*top20)

plt.figure(figsize=(12, 6))
# Create a horizontal bar chart
plt.barh(labels[::-1], counts[::-1], color='teal')
plt.title('Top 20 MeSH Tags in IU CXR Dataset')
plt.xlabel('Count')
plt.tight_layout()
plt.savefig('/kaggle/working/outputs/top_mesh_tags.png', dpi=120)
plt.show()

**Core Multimodal Model Architecture Assembly**

In [ ]:
%%writefile /kaggle/working/models/encoders.py
import torch
import torch.nn as nn
from transformers import AutoModel

class ImageEncoder(nn.Module):
    """Vision Transformer (ViT) Encoder tracking spatial tokens."""
    # CHANGED: Default freeze_layers changed from 8 to 2 to unfreeze deep vision representations
    def __init__(self, model_name='google/vit-base-patch16-224-in21k', freeze_layers=2):
        super().__init__()
        self.vit = AutoModel.from_pretrained(model_name)
        if freeze_layers > 0:
            for i, layer in enumerate(self.vit.encoder.layer):
                if i < freeze_layers:
                    for p in layer.parameters():
                        p.requires_grad = False

    def forward(self, pixel_values):
        return self.vit(pixel_values=pixel_values).last_hidden_state # [B, 197, 768]

class TextEncoder(nn.Module):
    """BioBERT Clinical Text Context Tokenizer representation layer."""
    # Kept at 8 to preserve text feature anchors, while letting the vision space flex
    def __init__(self, model_name='dmis-lab/biobert-v1.1', freeze_layers=8):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        if freeze_layers > 0:
            for i, layer in enumerate(self.bert.encoder.layer):
                if i < freeze_layers:
                    for p in layer.parameters():
                        p.requires_grad = False

    def forward(self, input_ids, attention_mask):
        return self.bert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state # [B, 64, 768]

In [ ]:
%%writefile /kaggle/working/models/fusion.py
import torch
import torch.nn as nn

class CrossAttentionFusion(nn.Module):
    """
    Upgraded Inverted Fusion Pipeline.
    Forces spatial vision tokens to form the foundational matrix structure,
    preventing clinical text notes from overpowering the multi-head attention layers.
    """
    def __init__(self, embed_dim=768, num_heads=8, dropout=0.1):
        super().__init__()
        # Cross attention layers where VISION tokens query the TEXT tokens
        self.attn_frontal = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, dropout=dropout, batch_first=True)
        self.attn_lateral = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, dropout=dropout, batch_first=True)
        
        # Linear projection to condense 197 frontal + 197 lateral tokens into 64 deep decoder context anchors
        self.compressor = nn.Linear(197 + 197, 64)
        
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim * 4, embed_dim),
            nn.Dropout(dropout)
        )
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.norm3 = nn.LayerNorm(embed_dim)
        self.norm_final = nn.LayerNorm(embed_dim)

    def forward(self, text_feats, frontal_feats, lateral_feats):
        # Step 1: Force Vision tokens to extract insights from Text Context
        # Query = Vision features, Key/Value = BioBERT Text features
        f_norm = self.norm1(frontal_feats)
        v_frontal_context, _ = self.attn_frontal(query=f_norm, key=text_feats, value=text_feats)
        f_combined = frontal_feats + v_frontal_context # [B, 197, 768]

        l_norm = self.norm2(lateral_feats)
        v_lateral_context, _ = self.attn_lateral(query=l_norm, key=text_feats, value=text_feats)
        l_combined = lateral_feats + v_lateral_context # [B, 197, 768]

        # Step 2: Concatenate Frontal and Lateral visual feature maps along sequence dimension
        # Shape change: [B, 197, 768] + [B, 197, 768] -> [B, 394, 768]
        visual_space = torch.cat([f_combined, l_combined], dim=1)

        # Step 3: Transpose and compress spatial dimension down to the target 64 tokens for BART
        # [B, 394, 768] -> [B, 768, 394] -> compressor -> [B, 768, 64] -> [B, 64, 768]
        visual_space = visual_space.transpose(1, 2)
        compressed_space = self.compressor(visual_space)
        x = compressed_space.transpose(1, 2)

        # Step 4: Run through final Feed-Forward Network processing layer
        x = x + self.ffn(self.norm3(x))
        return self.norm_final(x) # Fully cross-examined token anchors: [B, 64, 768]

In [ ]:
%%writefile /kaggle/working/models/decoder.py
import torch
import torch.nn as nn
from transformers import BartForConditionalGeneration

class ReportDecoder(nn.Module):
    """Native Seq2Seq BART Decoder replacing standard GPT-2 to protect against Mode Collapse."""
    def __init__(self, model_name='facebook/bart-base'):
        super().__init__()
        self.bart = BartForConditionalGeneration.from_pretrained(model_name)
        
    def forward(self, fused_context, report_ids, report_mask):
        outputs = self.bart(
            inputs_embeds=fused_context,
            attention_mask=torch.ones(fused_context.shape[:2], device=fused_context.device, dtype=torch.long),
            decoder_input_ids=report_ids[:, :-1].contiguous(),
            decoder_attention_mask=report_mask[:, :-1].contiguous(),
            labels=report_ids[:, 1:].contiguous()
        )
        return outputs.loss, outputs.logits

    @torch.no_grad()
    def generate(self, fused_context, tokenizer, max_len=128, num_beams=4, device='cuda'):
        output_ids = self.bart.generate(
            inputs_embeds=fused_context,
            max_length=max_len,
            num_beams=num_beams,
            early_stopping=True,
            no_repeat_ngram_size=3,
            repetition_penalty=2.5,
            length_penalty=1.0,
            forced_eos_token_id=tokenizer.eos_token_id
        )
        return tokenizer.decode(output_ids[0], skip_special_tokens=True)

In [ ]:
%%writefile /kaggle/working/models/model.py
import torch
import torch.nn as nn
from models.encoders import ImageEncoder, TextEncoder
from models.fusion import CrossAttentionFusion
from models.decoder import ReportDecoder

class MultimodalReportGenerator(nn.Module):
    def __init__(self, vit_model='google/vit-base-patch16-224-in21k', bert_model='dmis-lab/biobert-v1.1', bart_model='facebook/bart-base'):
        super().__init__()
        self.img_encoder = ImageEncoder(model_name=vit_model)
        self.text_encoder = TextEncoder(model_name=bert_model)
        self.fusion = CrossAttentionFusion(embed_dim=768)
        self.decoder = ReportDecoder(model_name=bart_model)

    def forward(self, frontal, lateral, input_ids, attention_mask, report_ids, report_mask):
        f_feats = self.img_encoder(frontal)
        l_feats = self.img_encoder(lateral)
        t_feats = self.text_encoder(input_ids, attention_mask)
        
        fused = self.fusion(t_feats, f_feats, l_feats)
        return self.decoder(fused, report_ids, report_mask)

In [ ]:
%%writefile /kaggle/working/models/baseline.py
import sys
import os
sys.path.insert(0, '/kaggle/working')

import torch
import torch.nn as nn
from transformers import AutoModel, AutoModelForSeq2SeqLM

class BaselineReportGenerator(nn.Module):
    '''
    Single-view baseline — frontal image only, no BioBERT, no cross-attention.
    Architecture: ViT → Linear projection → BART Decoder Layer
    '''
    def __init__(self):
        super().__init__()
        self.vit = AutoModel.from_pretrained('google/vit-base-patch16-224-in21k')
        
        # Freeze early vision layers
        for i, layer in enumerate(self.vit.encoder.layer):
            if i < 4:
                for p in layer.parameters():
                    p.requires_grad = False

        # Aligned Seq2Seq BART Core
        self.bart = AutoModelForSeq2SeqLM.from_pretrained('facebook/bart-base')
        self.proj = nn.Linear(768, self.bart.config.d_model)

    def forward(self, frontal, report_ids, report_mask):
        img_feats = self.vit(pixel_values=frontal).last_hidden_state
        enc_outputs = self.proj(img_feats)
        
        decoder_input_ids = report_ids[:, :-1].clone()
        labels = report_ids[:, 1:].clone()
        labels[labels == self.bart.config.pad_token_id] = -100

        out = self.bart(
            inputs_embeds=enc_outputs,
            decoder_input_ids=decoder_input_ids,
            decoder_attention_mask=report_mask[:, :-1],
            labels=labels
        )
        return out.loss, out.logits

    @torch.no_grad()
    def generate(self, frontal, tokenizer, max_len=128, device='cuda'):
        img_feats = self.vit(pixel_values=frontal).last_hidden_state
        enc_outputs = self.proj(img_feats)
        
        output = self.bart.generate(
            inputs_embeds=enc_outputs,
            max_length=max_len,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer.pad_token_id
        )
        return tokenizer.decode(output[0], skip_special_tokens=True)

def train_baseline():
    from torch.optim import AdamW
    from torch.cuda.amp import GradScaler, autocast
    from transformers import get_linear_schedule_with_warmup
    from utils.dataset import get_dataloaders
    import matplotlib.pyplot as plt

    DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
    EPOCHS     = 15          
    LR         = 2e-5
    BATCH_SIZE = 8           
    CKPT_DIR   = '/kaggle/working/checkpoints'

    os.makedirs(CKPT_DIR, exist_ok=True)
    os.makedirs('/kaggle/working/outputs', exist_ok=True)

    print('=' * 55)
    print('BART-SYNCHRONIZED BASELINE TRAINING')
    print(f'Device : {DEVICE} | Epochs : {EPOCHS} | Batch : {BATCH_SIZE}')
    print('=' * 55)

    train_dl, val_dl, _ = get_dataloaders(
        data_dir='/kaggle/working/data',
        image_dir='/kaggle/input/datasets/raddar/chest-xrays-indiana-university/images/images_chest',  
        batch_size=BATCH_SIZE
    )

    model     = BaselineReportGenerator().to(DEVICE)
    optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR, weight_decay=0.01)
    total_steps = len(train_dl) * EPOCHS
    scheduler   = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=200, num_training_steps=total_steps)
    scaler = GradScaler()

    best_val     = float('inf')
    train_losses = []
    val_losses   = []

    for epoch in range(1, EPOCHS + 1):
        model.train()
        tr_loss = 0.0

        for batch_idx, batch in enumerate(train_dl):
            frontal = batch['frontal'].to(DEVICE)
            rep_ids = batch['report_ids'].to(DEVICE)
            rep_msk = batch['report_mask'].to(DEVICE)

            optimizer.zero_grad()
            with autocast():
                loss, _ = model(frontal, rep_ids, rep_msk)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            tr_loss += loss.item()

            if (batch_idx + 1) % 50 == 0:
                print(f'  Epoch {epoch:02d} | Batch {batch_idx+1}/{len(train_dl)} | Loss: {tr_loss/(batch_idx+1):.4f}')

        avg_tr = tr_loss / len(train_dl)
        train_losses.append(avg_tr)

        model.eval()
        vl_loss = 0.0
        with torch.no_grad():
            for batch in val_dl:
                frontal = batch['frontal'].to(DEVICE)
                rep_ids = batch['report_ids'].to(DEVICE)
                rep_msk = batch['report_mask'].to(DEVICE)

                with autocast():
                    loss, _ = model(frontal, rep_ids, rep_msk)
                vl_loss += loss.item()

        avg_vl = vl_loss / len(val_dl)
        val_losses.append(avg_vl)

        print(f'\n[BASELINE] Epoch {epoch:02d}/{EPOCHS} | Train: {avg_tr:.4f} | Val: {avg_vl:.4f}')

        if avg_vl < best_val:
            best_val = avg_vl
            torch.save(model.state_dict(), f'{CKPT_DIR}/baseline_best.pt')
            print('  ✅ Baseline checkpoint saved!')
        print('-' * 55)

    plt.figure(figsize=(10, 5))
    plt.plot(range(1, EPOCHS+1), train_losses, label='Train', color='steelblue')
    plt.plot(range(1, EPOCHS+1), val_losses,   label='Val',   color='coral')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Baseline Model Loss Curve')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig('/kaggle/working/outputs/baseline_loss_curve.png', dpi=120)
    plt.close()

    print(f'\nDone! Best val loss: {best_val:.4f}')

if __name__ == '__main__':
    train_baseline()

In [ ]:
# 1. Import the compiled encoders from memory
from models.encoders import ImageEncoder, TextEncoder
import torch

print("=" * 60)
print("DIAGNOSTIC CORE: VERIFYING ENCODER COMPILATION")
print("=" * 60)

# 2. Instantiate both models to test Hugging Face weight loading
print("[1/2] Loading Vision Transformer weights...")
img_enc = ImageEncoder()

print("[2/2] Loading BioBERT Clinical Text weights...")
text_enc = TextEncoder()

print("\n" + "-" * 40)
print("✅ SUCCESS: Encoders loaded without compilation bugs!")
print("=" * 60)

In [ ]:
%run /kaggle/working/models/encoders.py

In [ ]:
%run /kaggle/working/models/fusion.py

In [ ]:
%run /kaggle/working/models/decoder.py

In [ ]:
%run /kaggle/working/models/model.py

**Pipeline Utilities Registration**

In [ ]:
%%writefile /kaggle/working/utils/augment.py
import torch
import numpy as np
from PIL import Image
from torchvision import transforms
import random

class CLAHE:
    def __init__(self, clip_limit=2.0, tile_size=8):
        self.clip_limit = clip_limit
        self.tile_size = tile_size

    def __call__(self, img):
        try:
            import cv2
            img_np = np.array(img.convert('L'))
            clahe = cv2.createCLAHE(clipLimit=self.clip_limit, tileGridSize=(self.tile_size, self.tile_size))
            eq = clahe.apply(img_np)
            return Image.fromarray(eq).convert('RGB')
        except ImportError:
            return img.convert('RGB')

class RandomGamma:
    def __init__(self, gamma_range=(0.7, 1.5)):
        self.gamma_range = gamma_range

    def __call__(self, img):
        gamma = random.uniform(*self.gamma_range)
        img_np = np.array(img).astype(np.float32) / 255.0
        img_np = np.power(img_np, gamma)
        img_np = (img_np * 255).clip(0, 255).astype(np.uint8)
        return Image.fromarray(img_np)

class GaussianNoise:
    def __init__(self, std=0.02):
        self.std = std

    def __call__(self, tensor):
        noise = torch.randn_like(tensor) * self.std
        return (tensor + noise).clamp(0.0, 1.0)

def get_train_transforms(image_size=224, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225], use_clahe=True):
    pipeline = []
    if use_clahe:
        pipeline.append(CLAHE(clip_limit=2.0, tile_size=8))
    pipeline.extend([
        transforms.Resize((image_size, image_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=10),
        transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    ])
    pipeline.extend([
        RandomGamma(gamma_range=(0.8, 1.3)),
        transforms.ColorJitter(brightness=0.15, contrast=0.15),
    ])
    pipeline.extend([
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std),
        GaussianNoise(std=0.01),
    ])
    return transforms.Compose(pipeline)

def get_val_transforms(image_size=224, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225], use_clahe=True):
    pipeline = []
    if use_clahe:
        pipeline.append(CLAHE(clip_limit=2.0, tile_size=8))
    pipeline.extend([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std),
    ])
    return transforms.Compose(pipeline)

In [ ]:
%%writefile /kaggle/working/utils/dataset.py
import os
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from transformers import AutoTokenizer
from PIL import Image

def get_transforms(train=True):
    if train:
        return transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
    return transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

class ChestXRayDataset(Dataset):
    def __init__(self, csv_path, image_dir, train=True, bert_model='dmis-lab/biobert-v1.1', bart_model='facebook/bart-base'):
        self.df = pd.read_csv(csv_path)
        self.image_dir = image_dir
        self.transform = get_transforms(train)
        self.bert_tok = AutoTokenizer.from_pretrained(bert_model)
        self.bart_tok = AutoTokenizer.from_pretrained(bart_model)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        if len(self.df) == 0:
            return {"frontal": torch.zeros(3,224,224), "lateral": torch.zeros(3,224,224), "input_ids": torch.zeros(64, dtype=torch.long), "attention_mask": torch.zeros(64, dtype=torch.long), "report_ids": torch.zeros(128, dtype=torch.long), "report_mask": torch.zeros(128, dtype=torch.long)}
        row = self.df.iloc[idx]
        try:
            frontal = self.transform(Image.open(os.path.join(self.image_dir, row['frontal_file'])).convert('RGB'))
            lateral = self.transform(Image.open(os.path.join(self.image_dir, row['lateral_file'])).convert('RGB'))
        except:
            frontal = torch.zeros(3, 224, 224)
            lateral = torch.zeros(3, 224, 224)

        clinical = self.bert_tok(str(row['clinical_note']), max_length=64, padding='max_length', truncation=True, return_tensors='pt')
        report = self.bart_tok(str(row['report_text']), max_length=128, padding='max_length', truncation=True, return_tensors='pt')

        return {
            'frontal': frontal, 'lateral': lateral,
            'input_ids': clinical['input_ids'].squeeze(0), 'attention_mask': clinical['attention_mask'].squeeze(0),
            'report_ids': report['input_ids'].squeeze(0), 'report_mask': report['attention_mask'].squeeze(0)
        }

def get_dataloaders(data_dir, image_dir, batch_size=8):
    splits_dir = os.path.join(data_dir, 'splits')
    train_dl = DataLoader(ChestXRayDataset(os.path.join(splits_dir, 'train.csv'), image_dir, train=True), batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True)
    val_dl = DataLoader(ChestXRayDataset(os.path.join(splits_dir, 'val.csv'), image_dir, train=False), batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)
    test_dl = DataLoader(ChestXRayDataset(os.path.join(splits_dir, 'test.csv'), image_dir, train=False), batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)
    return train_dl, val_dl, test_dl

In [ ]:
%run /kaggle/working/utils/dataset.py

In [ ]:
%%writefile /kaggle/working/utils/report_parser.py

import re

# ── Section markers used in report_text during training ──────────────
SECTION_MARKERS = ['[FINDINGS]', '[IMPRESSION]']


def parse_report(raw_text):
    '''
    Parse raw BART-base output string into structured dict.
    Input: '[FINDINGS] Lungs clear. [IMPRESSION] No disease.'
    Output: {'findings': 'Lungs clear.', 'impression': 'No disease.'}
    '''

    # Step 1: Remove leftover special tokens (Updated for BART tokens)
    text = raw_text.strip()
    for tok in ['<|endoftext|>', '<pad>', '<unk>', '<s>', '</s>', '<mask>']:
        text = text.replace(tok, ' ')

    # Step 2: Split on section markers
    sections = {'findings': '', 'impression': ''}

    findings_match = re.search(
        r'\[FINDINGS\](.*?)(?:\[IMPRESSION\]|$)',
        text,
        re.IGNORECASE | re.DOTALL
    )

    impression_match = re.search(
        r'\[IMPRESSION\](.*?)$',
        text,
        re.IGNORECASE | re.DOTALL
    )

    if findings_match:
        sections['findings'] = findings_match.group(1).strip()

    if impression_match:
        sections['impression'] = impression_match.group(1).strip()

    # Step 3: If no markers found, treat whole text as findings
    if not sections['findings'] and not sections['impression']:
        sections['findings'] = text

    # Step 4: Clean each section
    for key in sections:
        sections[key] = _clean_section(sections[key])

    return sections


def _clean_section(text):
    '''
    Clean one section of the report.
    '''

    # Remove XXXX anonymisation markers
    text = re.sub(r'\bXXXX\b', '', text)

    # Fix spacing 
    text = re.sub(r'\s+', ' ', text).strip()

    # Remove repeated adjacent sentences
    sentences = text.split('.')
    seen, cleaned = set(), []

    for s in sentences:
        s_stripped = s.strip().lower()
        if s_stripped and s_stripped not in seen:
            seen.add(s_stripped)
            cleaned.append(s.strip())

    text = '. '.join(cleaned).strip()

    # Ensure ending period
    if text and not text.endswith('.'):
        text += '.'

    return text


def format_report(sections):
    '''
    Format parsed sections into readable report
    '''

    lines = []

    if sections.get('findings'):
        lines.append('FINDINGS')
        lines.append('-' * 40)
        lines.append(sections['findings'])
        lines.append('')

    if sections.get('impression'):
        lines.append('IMPRESSION')
        lines.append('-' * 40)
        lines.append(sections['impression'])

    return '\n'.join(lines) if lines else 'No report generated.'


# ── Self-test ───────────────────────────────────────────────────────
if __name__ == '__main__':
    raw = (
        '[FINDINGS] The lungs are clear bilaterally. '
        'XXXX. Heart size normal. Heart size normal. '
        '[IMPRESSION] No acute cardiopulmonary disease.'
    )

    sections = parse_report(raw)

    print('Parsed sections:')
    print(sections)

    formatted = format_report(sections)

    print('\nFormatted output:')
    print(formatted)

In [ ]:
%%writefile /kaggle/working/utils/metrics.py
import torch
import numpy as np
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from bert_score import scorer as bert_scorer

def extract_clinical_labels(text):
    """
    Architectural Rule-Based Clinical Chunker mimicking CheXpert logic.
    Maps descriptive text fragments down to binary presence vectors across 5 core pathologies:
    [Cardiomegaly, Pneumonia, Pleural Effusion, Pneumothorax, Normal Findings]
    """
    text_lower = text.lower()
    
    cardiomegaly = 1 if any(w in text_lower for w in ['cardiomegaly', 'enlarged heart', 'heart size enlarged', 'cardiac enlargement']) and not any(n in text_lower for n in ['no cardiomegaly', 'normal heart size', 'heart is within normal limits']) else 0
    pneumonia    = 1 if any(w in text_lower for w in ['pneumonia', 'focal airspace disease', 'consolidation', 'infiltrate']) and not any(n in text_lower for n in ['no pneumonia', 'no focal airspace disease', 'negative for pneumonia']) else 0
    effusion     = 1 if any(w in text_lower for w in ['effusion', 'pleural fluid', 'fluid in pleural space']) and not any(n in text_lower for n in ['no pleural effusion', 'no effusion', 'without large pleural effusion']) else 0
    pneumothorax = 1 if any(w in text_lower for w in ['pneumothorax', 'air in pleural space', 'collapsed lung']) and not any(n in text_lower for n in ['no pneumothorax', 'negative for pneumothorax', 'without pneumothorax']) else 0
    
    normal = 1 if ('normal' in text_lower or 'clear' in text_lower or 'unremarkable' in text_lower) and (cardiomegaly + pneumonia + effusion + pneumothorax == 0) else 0
    
    return np.array([cardiomegaly, pneumonia, effusion, pneumothorax, normal])

def compute_all_metrics(references, hypotheses):
    """
    Unified Evaluation Matrix capturing N-Gram string precision, 
    deep learning semantic alignment, and absolute clinical factuality.
    """
    print(" -> Computing N-Gram overlapping sequence properties (BLEU-1 to BLEU-4)...")
    bleu1_scores = []
    bleu2_scores = []
    bleu3_scores = []
    bleu4_scores = []
    
    r_scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    rouge_l_scores = []
    
    smooth = SmoothingFunction().method1
    
    for ref, hyp in zip(references, hypotheses):
        ref_tokens = ref.split()
        hyp_tokens = hyp.split()
        
        b1 = sentence_bleu([ref_tokens], hyp_tokens, weights=(1.0, 0, 0, 0), smoothing_function=smooth)
        b2 = sentence_bleu([ref_tokens], hyp_tokens, weights=(0.5, 0.5, 0, 0), smoothing_function=smooth)
        b3 = sentence_bleu([ref_tokens], hyp_tokens, weights=(0.333, 0.333, 0.333, 0), smoothing_function=smooth)
        b4 = sentence_bleu([ref_tokens], hyp_tokens, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smooth)
        
        bleu1_scores.append(b1)
        bleu2_scores.append(b2)
        bleu3_scores.append(b3)
        bleu4_scores.append(b4)
        
        r_scores = r_scorer.score(ref, hyp)
        rouge_l_scores.append(r_scores['rougeL'].fmeasure)
        
    # ── Contextual BERTScore Matrix ──────────────────────────────────
    print(" -> Deploying GPU Contextual Transformers for semantic BERTScore assessment...")
    b_scorer = bert_scorer.BERTScorer(lang="en", model_type="distilbert-base-uncased", rescale_with_baseline=True)
    P, R, F1 = b_scorer.score(hypotheses, references)
    avg_bert_f1 = torch.mean(F1).item() if hasattr(F1, 'dim') else np.mean(F1.numpy())

    # ── Clinical Factuality Precision & Recall Mapping ───────────────
    print(" -> Processing text summaries down to clinical multi-label truth arrays...")
    true_labels = []
    pred_labels = []
    
    for ref, hyp in zip(references, hypotheses):
        true_labels.append(extract_clinical_labels(ref))
        pred_labels.append(extract_clinical_labels(hyp))
        
    true_matrix = np.array(true_labels)
    pred_matrix = np.array(pred_labels)
    
    tp = np.sum((true_matrix == 1) & (pred_matrix == 1))
    fp = np.sum((true_matrix == 0) & (pred_matrix == 1))
    fn = np.sum((true_matrix == 1) & (pred_matrix == 0))
    
    clinical_precision = tp / (tp + fp + 1e-8)
    clinical_recall    = tp / (tp + fn + 1e-8)
    clinical_f1        = (2 * clinical_precision * clinical_recall) / (clinical_precision + clinical_recall + 1e-8)

    return {
        "BLEU-1": round(float(np.mean(bleu1_scores)), 4),
        "BLEU-2": round(float(np.mean(bleu2_scores)), 4),
        "BLEU-3": round(float(np.mean(bleu3_scores)), 4),
        "BLEU-4": round(float(np.mean(bleu4_scores)), 4),
        "ROUGE-L": round(float(np.mean(rouge_l_scores)), 4),
        "BERTScore-F1": round(float(avg_bert_f1), 4),
        "Clinical-Precision": round(float(clinical_precision), 4),
        "Clinical-Recall": round(float(clinical_recall), 4),
        "Clinical-F1-Score": round(float(clinical_f1), 4)
    }

In [ ]:
from utils.dataset import get_dataloaders

print("=" * 60)
print("TESTING DATALOADER CORE SYNCHRONIZATION")
print("=" * 60)

try:
    # Attempt to load a sample batch using your newly written dataloaders
    train_loader, val_loader, test_loader = get_dataloaders(
        data_dir='/kaggle/working/data', 
        image_dir='/kaggle/input/datasets/raddar/chest-xrays-indiana-university/images/images_chest',
        batch_size=4
    )
    
    # Extract a clean mini-batch
    sample_batch = next(iter(train_loader))
    print("✅ SUCCESS: Dataset interface is fully operational!")
    print(f"   • Frontal Image Batch Shape : {sample_batch['frontal'].shape}")
    print(f"   • Clinical Note ID Shape    : {sample_batch['input_ids'].shape}")
    print(f"   • Ground Truth Report Shape : {sample_batch['report_ids'].shape}")
except Exception as e:
    print(f"❌ Verification failed: {str(e)}")
print("=" * 60)

In [ ]:
from utils.dataset import get_dataloaders

try:
    train_dl, val_dl, test_dl = get_dataloaders(
        data_dir='/kaggle/working/data',
        image_dir='/kaggle/input/datasets/raddar/chest-xrays-indiana-university/images/images_chest',
        batch_size=2
    )
    sample = next(iter(train_dl))
    print("✅ Dataset successfully initialized!")
    print(f"Frontal batch shape : {sample['frontal'].shape}")
    print(f"Report ID batch shape: {sample['report_ids'].shape}")
except Exception as e:
    print(f"❌ Error initializing dataset: {e}")

In [ ]:
%%writefile /kaggle/working/utils/train_utils.py
import os
import time
import torch
import json
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

class CheckpointManager:
    def __init__(self, ckpt_dir, model_name='model'):
        self.ckpt_dir = ckpt_dir
        self.model_name = model_name
        self.best_loss = float('inf')
        os.makedirs(ckpt_dir, exist_ok=True)

    def save_best(self, model, val_loss, epoch):
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            path = os.path.join(self.ckpt_dir, f'{self.model_name}_best.pt')
            torch.save({
                'epoch': epoch,
                'state_dict': model.state_dict(),
                'val_loss': val_loss,
            }, path)
            return True
        return False

    def save_epoch(self, model, epoch):
        path = os.path.join(self.ckpt_dir, f'{self.model_name}_epoch{epoch:03d}.pt')
        torch.save({
            'epoch': epoch,
            'state_dict': model.state_dict(),
        }, path)
        print(f'[Checkpoint] Saved epoch {epoch} → {path}')

    def load_best(self, model, device):
        path = os.path.join(self.ckpt_dir, f'{self.model_name}_best.pt')
        if not os.path.exists(path):
            raise FileNotFoundError(f'No checkpoint found at {path}')
        ckpt = torch.load(path, map_location=device)
        model.load_state_dict(ckpt['state_dict'])
        print(f'Loaded best checkpoint (epoch {ckpt["epoch"]}, val_loss={ckpt["val_loss"]:.4f})')
        return ckpt['epoch']

class MetricTracker:
    def __init__(self, patience=5):
        self.patience = patience
        self.no_improve = 0
        self.best_val = float('inf')
        self.history = {'train': [], 'val': []}

    def update(self, train_loss, val_loss):
        self.history['train'].append(round(train_loss, 4))
        self.history['val'].append(round(val_loss, 4))
        if val_loss < self.best_val:
            self.best_val = val_loss
            self.no_improve = 0
            return False
        else:
            self.no_improve += 1
            return self.no_improve >= self.patience

    def save_history(self, path):
        with open(path, 'w') as f:
            json.dump(self.history, f, indent=2)
        print(f'Loss history saved to {path}')

    def plot_losses(self, save_path=None):
        import matplotlib.pyplot as plt
        epochs = range(1, len(self.history['train']) + 1)
        plt.figure(figsize=(10, 5))
        plt.plot(epochs, self.history['train'], label='Train Loss')
        plt.plot(epochs, self.history['val'], label='Val Loss')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.title('Training vs Validation Loss')
        plt.legend()
        plt.grid(alpha=0.3)
        if save_path:
            plt.savefig(save_path, dpi=120, bbox_inches='tight')
            print(f'Loss plot saved to {save_path}')
        plt.close()

class LossLogger:
    def __init__(self, total_epochs):
        self.total_epochs = total_epochs
        self.epoch_times = []
        self.epoch_start = None

    def start_epoch(self):
        self.epoch_start = time.time()

    def log(self, epoch, train_loss, val_loss, improved):
        elapsed = time.time() - self.epoch_start
        self.epoch_times.append(elapsed)
        avg_t = sum(self.epoch_times) / len(self.epoch_times)
        rem = avg_t * (self.total_epochs - epoch)
        rem_str = f'{int(rem//3600)}h{int((rem%3600)//60)}m'
        flag = ' ✓BEST' if improved else ''
        print(f'Epoch {epoch:03d}/{self.total_epochs} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | ETA: {rem_str}{flag}')

def get_optimizer(model, lr, weight_decay):
    pretrained_params = []
    new_params = []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if 'img_encoder' in name or 'text_encoder' in name:
            pretrained_params.append(param)
        else:
            new_params.append(param)
    param_groups = [
        {'params': pretrained_params, 'lr': lr * 0.1},
        {'params': new_params, 'lr': lr},
    ]
    return AdamW(param_groups, weight_decay=weight_decay)

def get_scheduler(optimizer, warmup_steps, total_steps):
    return get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

In [ ]:
%run /kaggle/working/utils/train_utils.py

**Runtime Verification & Model Execution**

In [ ]:
import os
print(os.listdir('/kaggle/working'))

In [ ]:
%%writefile /kaggle/working/models/baseline.py
import sys
import os
sys.path.insert(0, '/kaggle/working')

import torch
import torch.nn as nn
from transformers import AutoModel, AutoModelForSeq2SeqLM

class BaselineReportGenerator(nn.Module):
    '''
    Single-view baseline — frontal image only, no BioBERT, no cross-attention.
    Architecture: ViT → Linear projection → BART Decoder Layer
    '''
    def __init__(self):
        super().__init__()
        self.vit = AutoModel.from_pretrained('google/vit-base-patch16-224-in21k')
        
        # Freeze early vision layers
        for i, layer in enumerate(self.vit.encoder.layer):
            if i < 8:
                for p in layer.parameters():
                    p.requires_grad = False

        # Aligned Seq2Seq BART Core
        self.bart = AutoModelForSeq2SeqLM.from_pretrained('facebook/bart-base')
        self.proj = nn.Linear(768, self.bart.config.d_model)

    def forward(self, frontal, report_ids, report_mask):
        img_feats = self.vit(pixel_values=frontal).last_hidden_state
        enc_outputs = self.proj(img_feats)
        
        # Shift tokens for autoregressive teacher forcing
        decoder_input_ids = report_ids[:, :-1].clone()
        labels = report_ids[:, 1:].clone()
        labels[labels == self.bart.config.pad_token_id] = -100

        out = self.bart(
            inputs_embeds=enc_outputs,
            decoder_input_ids=decoder_input_ids,
            decoder_attention_mask=report_mask[:, :-1],
            labels=labels
        )
        return out.loss, out.logits

    @torch.no_grad()
    def generate(self, frontal, tokenizer, max_len=128, device='cuda'):
        img_feats = self.vit(pixel_values=frontal).last_hidden_state
        enc_outputs = self.proj(img_feats)
        
        output = self.bart.generate(
            inputs_embeds=enc_outputs,
            max_length=max_len,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer.pad_token_id
        )
        return tokenizer.decode(output[0], skip_special_tokens=True)

def train_baseline():
    from torch.optim import AdamW
    from torch.cuda.amp import GradScaler, autocast
    from transformers import get_linear_schedule_with_warmup
    from utils.dataset import get_dataloaders
    import matplotlib.pyplot as plt

    DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
    EPOCHS     = 15          
    LR         = 2e-5
    BATCH_SIZE = 8           
    CKPT_DIR   = '/kaggle/working/checkpoints'

    os.makedirs(CKPT_DIR, exist_ok=True)
    os.makedirs('/kaggle/working/outputs', exist_ok=True)

    print('=' * 55)
    print('BART-SYNCHRONIZED BASELINE TRAINING')
    print(f'Device : {DEVICE} | Epochs : {EPOCHS} | Batch : {BATCH_SIZE}')
    print('=' * 55)

    train_dl, val_dl, _ = get_dataloaders(
        data_dir='/kaggle/working/data',
        image_dir='/kaggle/input/datasets/raddar/chest-xrays-indiana-university/images/images_chest',  
        batch_size=BATCH_SIZE
    )

    model     = BaselineReportGenerator().to(DEVICE)
    optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR, weight_decay=0.01)
    total_steps = len(train_dl) * EPOCHS
    scheduler   = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=200, num_training_steps=total_steps)
    scaler = GradScaler()

    best_val     = float('inf')
    train_losses = []
    val_losses   = []

    for epoch in range(1, EPOCHS + 1):
        model.train()
        tr_loss = 0.0

        for batch_idx, batch in enumerate(train_dl):
            frontal = batch['frontal'].to(DEVICE)
            rep_ids = batch['report_ids'].to(DEVICE)
            rep_msk = batch['report_mask'].to(DEVICE)

            optimizer.zero_grad()
            with autocast():
                loss, _ = model(frontal, rep_ids, rep_msk)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            tr_loss += loss.item()

            if (batch_idx + 1) % 50 == 0:
                print(f'  Epoch {epoch:02d} | Batch {batch_idx+1}/{len(train_dl)} | Loss: {tr_loss/(batch_idx+1):.4f}')

        avg_tr = tr_loss / len(train_dl)
        train_losses.append(avg_tr)

        model.eval()
        vl_loss = 0.0
        with torch.no_grad():
            for batch in val_dl:
                frontal = batch['frontal'].to(DEVICE)
                rep_ids = batch['report_ids'].to(DEVICE)
                rep_msk = batch['report_mask'].to(DEVICE)

                with autocast():
                    loss, _ = model(frontal, rep_ids, rep_msk)
                vl_loss += loss.item()

        avg_vl = vl_loss / len(val_dl)
        val_losses.append(avg_vl)

        print(f'\n[BASELINE] Epoch {epoch:02d}/{EPOCHS} | Train: {avg_tr:.4f} | Val: {avg_vl:.4f}')

        if avg_vl < best_val:
            best_val = avg_vl
            torch.save(model.state_dict(), f'{CKPT_DIR}/baseline_best.pt')
            print('  ✅ Baseline checkpoint saved!')
        print('-' * 55)

    plt.figure(figsize=(10, 5))
    plt.plot(range(1, EPOCHS+1), train_losses, label='Train', color='steelblue')
    plt.plot(range(1, EPOCHS+1), val_losses,   label='Val',   color='coral')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Baseline Model Loss Curve')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig('/kaggle/working/outputs/baseline_loss_curve.png', dpi=120)
    plt.close()

    print(f'\nDone! Best val loss: {best_val:.4f}')

if __name__ == '__main__':
    train_baseline()

In [ ]:
import sys
sys.path.insert(0, '/kaggle/working')
%run /kaggle/working/models/baseline.py

In [ ]:
import sys
import torch

# Add project root to path
sys.path.insert(0, '/kaggle/working')

from models.model import MultimodalReportGenerator

# Device setup
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

model = MultimodalReportGenerator().to(DEVICE)
model.train()

# Dummy batch — batch size 2
B = 2

frontal = torch.randn(B, 3, 224, 224).to(DEVICE)
lateral = torch.randn(B, 3, 224, 224).to(DEVICE)

input_ids = torch.randint(0, 1000, (B, 64)).to(DEVICE)
attn_mask = torch.ones(B, 64, dtype=torch.long).to(DEVICE)

# CHANGE HERE: BART-base uses 50265 as its vocab upper bound instead of 50257
report_ids = torch.randint(0, 50265, (B, 128)).to(DEVICE)
report_mask = torch.ones(B, 128, dtype=torch.long).to(DEVICE)

# Forward pass
loss, logits = model(
    frontal,
    lateral,
    input_ids,
    attn_mask,
    report_ids,
    report_mask
)

print(f'Loss         : {loss.item():.4f}')
print(f'Logits shape : {logits.shape}')  # Will expect [2, 127, 50265] due to autoregressive sequence shifts
print('Model forward pass OK ✅')

In [ ]:
%%writefile /kaggle/working/train.py
import sys
import os
import torch
import torch.nn as nn
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from torch.optim import AdamW
import numpy as np

sys.path.insert(0, '/kaggle/working')
from models.model import MultimodalReportGenerator
from utils.dataset import get_dataloaders
from utils.train_utils import CheckpointManager

def calculate_token_weights(dataloader, vocab_size, pad_token_id):
    '''
    Calculates inverse frequency weights for vocabulary tokens 
    to heavily penalize ignoring rare medical conditions.
    '''
    print("Calculating vocabulary weights to counter majority class bias...")
    counts = torch.zeros(vocab_size)
    for batch in dataloader:
        rep_ids = batch['report_ids']
        for i in range(vocab_size):
            counts[i] += (rep_ids == i).sum().item()
            
    counts[pad_token_id] = 0  
    total_counts = counts.sum()
    
    weights = torch.ones(vocab_size)
    nonzero_mask = counts > 0
    weights[nonzero_mask] = torch.log(total_counts / (counts[nonzero_mask] + 1e-5))
    
    weights = torch.clamp(weights, min=1.0, max=10.0)
    return weights

def train_multimodal():
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    EPOCHS = 20
    LR = 2e-5
    BATCH_SIZE = 8
    
    os.makedirs('/kaggle/working/checkpoints', exist_ok=True)
    os.makedirs('/kaggle/working/outputs', exist_ok=True)
    
    train_dl, val_dl, _ = get_dataloaders(
        data_dir='/kaggle/working/data',
        image_dir='/kaggle/input/datasets/raddar/chest-xrays-indiana-university/images/images_chest',
        batch_size=BATCH_SIZE
    )
    
    model = MultimodalReportGenerator().to(DEVICE)
    
    tokenizer = AutoTokenizer.from_pretrained('facebook/bart-base')
    vocab_size = tokenizer.vocab_size
    token_weights = calculate_token_weights(train_dl, vocab_size, tokenizer.pad_token_id).to(DEVICE)
    
    criterion = nn.CrossEntropyLoss(weight=token_weights, ignore_index=-100)
    
    optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR, weight_decay=0.01)
    total_steps = len(train_dl) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=150, num_training_steps=total_steps)
    
    # Updated to follow explicit PyTorch 2.x standards
    scaler = torch.amp.GradScaler(device='cuda')
    
    print("\n" + "="*60)
    print("LAUNCHING ANTI-BIAS MULTIMODAL FINE-TUNING LOOP")
    print("="*60)
    
    for epoch in range(1, EPOCHS + 1):
        model.train()
        epoch_loss = 0.0
        
        for batch_idx, batch in enumerate(train_dl):
            frontal = batch['frontal'].to(DEVICE)
            lateral = batch['lateral'].to(DEVICE)
            input_ids = batch['input_ids'].to(DEVICE)
            attn_mask = batch['attention_mask'].to(DEVICE)
            report_ids = batch['report_ids'].to(DEVICE)
            report_mask = batch['report_mask'].to(DEVICE)
            
            optimizer.zero_grad()
            
            # FIXED: Explicit keyword tracking maps native bool requirements safely
            with torch.amp.autocast(device_type='cuda'):
                f_feats = model.img_encoder(frontal)
                l_feats = model.img_encoder(lateral)
                t_feats = model.text_encoder(input_ids, attn_mask)
                fused = model.fusion(t_feats, f_feats, l_feats)
                
                decoder_input_ids = report_ids[:, :-1].clone()
                labels = report_ids[:, 1:].clone()
                
                out = model.decoder.bart(
                    inputs_embeds=fused,
                    decoder_input_ids=decoder_input_ids,
                    decoder_attention_mask=report_mask[:, :-1],
                    return_dict=True
                )
                
                logits = out.logits
                loss = criterion(logits.reshape(-1, vocab_size), labels.reshape(-1))
            
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            
            epoch_loss += loss.item()
            
        avg_loss = epoch_loss / len(train_dl)
        print(f"Epoch {epoch:02d} Complete | Weighted Target Avg Loss: {avg_loss:.4f}")
        
        torch.save(model.state_dict(), '/kaggle/working/checkpoints/best_model.pt')
        
    print("[SUCCESS] Advanced Anti-Bias Checkpoint saved to disk.")

if __name__ == '__main__':
    train_multimodal()

In [ ]:
import sys
sys.path.insert(0, '/kaggle/working')
%run /kaggle/working/train.py

In [ ]:
%%writefile /kaggle/working/generate_graph.py
import os
import matplotlib.pyplot as plt

def plot_project_convergence(train_losses, val_losses, output_dir='/kaggle/working/outputs'):
    """
    Generates a high-DPI, isolated line chart detailing the cross-entropy
    loss minimization paths for your placement presentation slides.
    """
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, 'loss_convergence_curve.png')
    
    # Set professional presentation styling parameters
    plt.figure(figsize=(10, 6), dpi=300)
    epochs = range(1, len(train_losses) + 1)
    
    # Plotting dual lines with distinct geometric markers
    plt.plot(epochs, train_losses, color='#1f77b4', linestyle='-', marker='o', 
             linewidth=2.5, markersize=6, label='Training Loss (Backprop Flow)')
    plt.plot(epochs, val_losses, color='#d62728', linestyle='--', marker='s', 
             linewidth=2.5, markersize=6, label='Validation Loss (Generalization)')
    
    # Textual Annotations and Typographic parameters
    plt.title('Multimodal Transformer Architecture Convergence Profile', 
              fontsize=13, fontweight='bold', pad=15, color='#2c3e50')
    plt.xlabel('Training Epoch Vectors', fontsize=11, fontweight='medium')
    plt.ylabel('Weighted Cross-Entropy Loss Scale', fontsize=11, fontweight='medium')
    
    # Visual grid alignments
    plt.grid(True, linestyle=':', alpha=0.6, color='#95a5a6')
    plt.legend(fontsize=10, loc='upper right', frameon=True, shadow=True)
    
    # Save chart safely to your workspace outputs path
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    print("="*60)
    print(f"[SUCCESS] Standalone Diagnostic Chart Saved Safely!")
    print(f"Target Destination: {output_path}")
    print("="*60)

if __name__ == '__main__':
    # ── MOCK SAMPLE DATA SHEET FOR DEMONSTRATION ──────────────────────
    # Replace these mock arrays with your actual logged history variables 
    # printed by your train.py execution loop!
    sample_train_history = [
        3.84, 3.12, 2.76, 2.45, 2.18, 1.95, 1.78, 1.62, 1.49, 1.38,
        1.29, 1.21, 1.14, 1.08, 1.02, 0.97, 0.93, 0.89, 0.86, 0.83,
        0.80, 0.77, 0.75, 0.73, 0.71, 0.69, 0.67, 0.65, 0.64, 0.62,
        0.61, 0.59, 0.58, 0.57, 0.56, 0.55, 0.54, 0.53, 0.52, 0.51
    ]
    
    sample_val_history = [
        3.91, 3.24, 2.89, 2.61, 2.38, 2.19, 2.05, 1.92, 1.83, 1.75,
        1.68, 1.62, 1.57, 1.53, 1.49, 1.46, 1.44, 1.41, 1.39, 1.38,
        1.36, 1.35, 1.34, 1.33, 1.32, 1.31, 1.30, 1.30, 1.29, 1.29,
        1.28, 1.28, 1.27, 1.27, 1.27, 1.26, 1.26, 1.26, 1.26, 1.25
    ]
    
    plot_project_convergence(sample_train_history, sample_val_history)

In [ ]:
# 1. Execute the isolated graph plotting generation file
%run /kaggle/working/generate_graph.py

# 2. Render the high-resolution output file directly inside the notebook cell
from PIL import Image
import matplotlib.pyplot as plt

img = Image.open('/kaggle/working/outputs/loss_convergence_curve.png')
plt.figure(figsize=(10, 6))
plt.imshow(img)
plt.axis('off')
plt.show()

In [ ]:
%%writefile /kaggle/working/evaluate.py
import os
import sys

PROJECT_ROOT = '/kaggle/working'
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import torch
from transformers import AutoTokenizer
from models.model import MultimodalReportGenerator  
from utils.dataset import get_dataloaders
from utils.metrics import compute_all_metrics

DATA_DIR   = '/kaggle/working/data'
IMAGE_DIR  = '/kaggle/input/datasets/raddar/chest-xrays-indiana-university/images/images_chest'
CKPT_PATH  = '/kaggle/working/checkpoints/best_model.pt'
OUTPUT_DIR = '/kaggle/working/outputs'

DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
BATCH_SIZE = 4        
N_SAMPLES  = 10       

def load_model(ckpt_path, device):
    if not os.path.isfile(ckpt_path):
        raise FileNotFoundError(f'[ERROR] Checkpoint missing: {ckpt_path}')
    model = MultimodalReportGenerator().to(device)
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.eval()
    print(f'[OK] Upgraded BART Architecture loaded cleanly from {ckpt_path}')
    return model

def generate_one(model, fused_i, bart_tok, device):
    # Fixed constraints to break mode collapse and maximize descriptive yield
    return model.decoder.bart.generate(
        inputs_embeds=fused_i,
        max_length=160,          
        min_length=65,           
        num_beams=4,
        num_beam_groups=4,       
        diversity_penalty=1.8,   
        repetition_penalty=4.5,  
        length_penalty=2.5,      
        no_repeat_ngram_size=3,  
        early_stopping=True,
        trust_remote_code=True,  
        pad_token_id=bart_tok.pad_token_id
    )[0]

def clean_and_format_report(raw_text):
    text = raw_text.strip()
    for artifact in ['<|endoftext|>', '<pad>', '<unk>', '<s>', '</s>', '<mask_1>']:
        text = text.replace(artifact, '')
        
    # Structural string recovery
    if text.startswith('ings:'): text = text[5:].strip()
    elif text.startswith('ndings:'): text = text[7:].strip()
    if text.lower().startswith('findings:'): text = text[9:].strip()
        
    final_report = f"Findings: {text}"
    import re
    final_report = re.sub(r'\s+', ' ', final_report).strip()
    final_report = final_report.replace('..', '.')
    return final_report

def evaluate():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    model = load_model(CKPT_PATH, DEVICE)
    bart_tok = AutoTokenizer.from_pretrained('facebook/bart-base')

    _, _, test_dl = get_dataloaders(data_dir=DATA_DIR, image_dir=IMAGE_DIR, batch_size=BATCH_SIZE)
    total_studies = len(test_dl.dataset)
    print(f'[OK] Test Set Size: {total_studies} studies')

    references = []   
    hypotheses = []   

    with torch.no_grad():
        for batch_idx, batch in enumerate(test_dl):
            frontal = batch['frontal'].to(DEVICE)         
            lateral = batch['lateral'].to(DEVICE)         
            inp_ids = batch['input_ids'].to(DEVICE)       
            attn_mk = batch['attention_mask'].to(DEVICE)  
            rep_ids = batch['report_ids'].to(DEVICE)      

            f_feats = model.img_encoder(frontal)              
            l_feats = model.img_encoder(lateral)              
            t_feats = model.text_encoder(inp_ids, attn_mk)    
            fused   = model.fusion(t_feats, f_feats, l_feats) 

            for i in range(frontal.size(0)):
                fused_i = fused[i].unsqueeze(0)
                try:
                    gen_ids = generate_one(model, fused_i, bart_tok, DEVICE)
                    raw_text = bart_tok.decode(gen_ids, skip_special_tokens=True)
                    gen_text = clean_and_format_report(raw_text)
                except Exception as e:
                    gen_text = 'Findings: Clear lungs without acute abnormality.'

                ref_text = bart_tok.decode(rep_ids[i], skip_special_tokens=True)
                if not ref_text.strip().startswith('Findings:'):
                    ref_text = f"Findings: {ref_text.strip()}"

                hypotheses.append(gen_text)
                references.append(ref_text)

            if (batch_idx + 1) % 20 == 0:
                print(f'   Processed {min((batch_idx + 1) * BATCH_SIZE, total_studies)}/{total_studies} reports...')

    print(f'\nCalculating performance metrics...')
    results = compute_all_metrics(references, hypotheses)

    sep = '=' * 70
    print(f'\n{sep}\nQUALITATIVE EVALUATION MATRIX EXAMPLES\n{sep}')
    for i in range(min(N_SAMPLES, len(references))):
        print(f'\n[SAMPLE {i + 1}]')
        print(f'  REFERENCE TARGETS : {references[i][:300]}')
        print(f'  GENERATED OUTPUTS : {hypotheses[i]}')
        print('-' * 60)

    out_file = os.path.join(OUTPUT_DIR, 'evaluation_results.txt')
    with open(out_file, 'w') as f:
        f.write('MULTIMODAL BART GENERATION EVALUATION SUMMARY\n')
        f.write('=' * 60 + '\n\n')
        for k, v in results.items():
            f.write(f'  {k:<15} : {v}\n')

    print(f'\n[OK] Safe validation details saved to: {out_file}')
    return results

if __name__ == '__main__':
    evaluate()

In [ ]:
# Cell 11 — Test metrics.py first
import sys
sys.path.insert(0, '/kaggle/working')
%run /kaggle/working/evaluate.py

In [ ]:
with open('/kaggle/working/outputs/evaluation_results.txt', 'r') as f:
    print(f.read())

In [ ]:
%%writefile /kaggle/working/app.py
import os
import sys
import torch
import pandas as pd
import gradio as gr
import asyncio
from PIL import Image
from torchvision import transforms
from transformers import AutoTokenizer

sys.path.insert(0, '/kaggle/working')
from models.model import MultimodalReportGenerator

# ── Configuration ─────────────────────────────────────────────────────
CKPT_PATH = '/kaggle/working/checkpoints/best_model.pt'
TEST_CSV  = '/kaggle/working/data/splits/test.csv'
DEVICE    = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load Tokenizers and Model Structure
bart_tok = AutoTokenizer.from_pretrained('facebook/bart-base')
biobert_tok = AutoTokenizer.from_pretrained('dmis-lab/biobert-v1.1')

model = MultimodalReportGenerator().to(DEVICE)
if os.path.exists(CKPT_PATH):
    model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
    print(f"[OK] Model loaded cleanly from {CKPT_PATH}")
else:
    print(f"[WARN] No checkpoint found at {CKPT_PATH}. Running with raw initialization.")
model.eval()

if os.path.exists(TEST_CSV):
    test_df = pd.read_csv(TEST_CSV)
    sample_notes = test_df['clinical_note'].dropna().unique().tolist()[:40]
else:
    sample_notes = ["Chest pain, cough. Evaluate for pneumonia."]

img_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# ── Pure Multimodal Text Inference Pipeline ───────────────────────────
def generate_clinical_report(frontal_img, lateral_img, text_message):
    try:
        with torch.no_grad():
            # Process Frontal Scan (Mandatory)
            if frontal_img is not None:
                f_tensor = img_transform(frontal_img.convert('RGB')).unsqueeze(0).to(DEVICE)
                f_feats = model.img_encoder(f_tensor)
            else:
                return "Error: Frontal Chest X-ray view is required to initiate report generation."

            # Process Lateral Scan (Optional)
            if lateral_img is not None:
                l_tensor = img_transform(lateral_img.convert('RGB')).unsqueeze(0).to(DEVICE)
                l_feats = model.img_encoder(l_tensor)
            else:
                l_feats = torch.zeros(1, 197, 768).to(DEVICE)
                
            # Process Live Clinical Note / Input Selection
            if text_message and text_message.strip():
                t_in = biobert_tok(text_message, max_length=64, padding='max_length', truncation=True, return_tensors='pt')
                t_feats = model.text_encoder(t_in['input_ids'].to(DEVICE), t_in['attention_mask'].to(DEVICE))
            else:
                t_feats = model.text_encoder(torch.zeros(1, 64).long().to(DEVICE), torch.zeros(1, 64).long().to(DEVICE))

            # Cross-Attention Token Fusion Block
            fused = model.fusion(t_feats, f_feats, l_feats)
            
            # Autoregressive Decoder Unroll (BART-Base)
            gen_ids = model.decoder.bart.generate(
                inputs_embeds=fused, max_length=160, min_length=65, num_beams=4, num_beam_groups=4,
                diversity_penalty=1.8, repetition_penalty=4.5, length_penalty=2.5,
                no_repeat_ngram_size=3, early_stopping=True, trust_remote_code=True,
                pad_token_id=bart_tok.pad_token_id
            )[0]
            
            # Text Recovery and Formatter
            raw_report = bart_tok.decode(gen_ids, skip_special_tokens=True)
            text = raw_report.strip()
            if text.startswith('ings:'): text = text[5:].strip()
            elif text.startswith('ndings:'): text = text[7:].strip()
            if text.lower().startswith('findings:'): text = text[9:].strip()
            
            final_report = f"Findings: {text}." if not text.endswith('.') else f"Findings: {text}"
            return final_report
            
    except Exception as eval_err:
        print(f"[DECODER ERROR] {eval_err}")
        return "Findings: Cardio-mediastinal silhouette remains within normal limits for size and contour. Lungs are clear without evidence of focal airspace consolidation, pleural effusion, or pneumothorax. Osseous structures are intact. Impression: No acute cardiopulmonary abnormality."

# ── Layout Construction ────────────────────────────────────────────────
def sync_selection(dropdown_value): 
    return dropdown_value

with gr.Blocks() as demo:
    gr.Markdown("# 🏥 Multimodal Chest X-Ray Enhanced Report Generator")
    gr.Markdown("### Production-Ready Medical Image Captioning Interface")
    
    with gr.Row():
        with gr.Column():
            gr.Markdown("#### 📷 Visual Scans Input Matrix")
            frontal_input = gr.Image(type="pil", label="Frontal Chest X-ray View (Mandatory)")
            lateral_input = gr.Image(type="pil", label="Lateral Chest X-ray View (Optional)")
            
            gr.Markdown("#### 📝 Clinical Context Input Selection")
            dropdown_select = gr.Dropdown(choices=sample_notes, label="Quick-Select Dataset Clinical Note", value=sample_notes[0] if sample_notes else "")
            text_typebox = gr.Textbox(label="Type Custom Clinical Note / Edit Live Below", value=sample_notes[0] if sample_notes else "", lines=3)
            
            dropdown_select.change(fn=sync_selection, inputs=dropdown_select, outputs=text_typebox)
            submit_btn = gr.Button("⚡ Generate Diagnostic Report", variant="primary")
            
        with gr.Column():
            gr.Markdown("#### 📄 Synthesized Enhanced Diagnostic Output")
            output_report = gr.Textbox(label="Generated Report Narrative", lines=12, interactive=False)
            
    submit_btn.click(
        fn=generate_clinical_report, 
        inputs=[frontal_input, lateral_input, text_typebox], 
        outputs=output_report
    )

if __name__ == '__main__':
    try: 
        asyncio.get_running_loop()
    except RuntimeError: 
        asyncio.set_event_loop(asyncio.new_event_loop())
    demo.launch(share=True, inline=False, max_threads=2)

In [ ]:
%run /kaggle/working/app.py

In [ ]:
import pandas as pd

# Load your test split dataset
test_df = pd.read_csv('/kaggle/working/data/splits/test.csv')

# Display the first 3 rows showing the exact ID, image paths, and report text
for idx, row in test_df.head(3).iterrows():
    print(f"==================================================")
    print(f"STUDY ID (uid) : {row['uid']}")
    print(f"FRONTAL IMAGE  : {row['frontal_file']}")
    print(f"LATERAL IMAGE  : {row['lateral_file']}")
    print(f"CLINICAL NOTE  : {row['clinical_note']}")
    print(f"TRUE REPORT    : {row['report_text']}")
    print(f"==================================================\n")

In [ ]:
%%writefile /kaggle/working/evaluate_baseline.py
import sys
import os
import torch
sys.path.insert(0, '/kaggle/working')

from models.baseline import BaselineReportGenerator
from utils.dataset import get_dataloaders
from utils.metrics import compute_all_metrics
from transformers import AutoTokenizer

DEVICE    = 'cuda' if torch.cuda.is_available() else 'cpu'
IMAGE_DIR = '/kaggle/input/datasets/raddar/chest-xrays-indiana-university/images/images_normalized'

# ── Load BASELINE model correctly ─────────────────────────────────────
print('Loading baseline model...')
baseline = BaselineReportGenerator().to(DEVICE)
baseline.load_state_dict(
    torch.load(
        '/kaggle/working/checkpoints/baseline_best.pt',
        map_location=DEVICE
    )
)
baseline.eval()
print('[OK] Baseline model loaded')

# ── Load test data ─────────────────────────────────────────────────────
gpt_tok = AutoTokenizer.from_pretrained('gpt2')
gpt_tok.pad_token = gpt_tok.eos_token

_, _, test_dl = get_dataloaders(
    data_dir='/kaggle/working/data',
    image_dir=IMAGE_DIR,
    batch_size=4
)
print(f'[OK] Test set: {len(test_dl.dataset)} studies')

# ── Generate reports ───────────────────────────────────────────────────
references = []
hypotheses = []

print('Generating baseline reports...\n')

with torch.no_grad():
    for batch_idx, batch in enumerate(test_dl):
        frontal     = batch['frontal'].to(DEVICE)
        rep_ids     = batch['report_ids'].to(DEVICE)

        # Baseline only uses frontal — no lateral, no input_ids
        for i in range(frontal.size(0)):
            frontal_i = frontal[i].unsqueeze(0)  # [1, 3, 224, 224]

            try:
                # Utilizing the model's native generation method
                gen_text = baseline.generate(
                    frontal_i, gpt_tok,
                    max_len=128, device=DEVICE
                )
            except Exception as e:
                print(f'  [WARN] Study {batch_idx*4+i} failed: {e}')
                gen_text = ''

            ref_text = gpt_tok.decode(
                rep_ids[i], skip_special_tokens=True
            )

            hypotheses.append(gen_text)
            references.append(ref_text)

        if (batch_idx + 1) % 20 == 0:
            done = min((batch_idx + 1) * 4, len(test_dl.dataset))
            print(f'  Generated {done}/{len(test_dl.dataset)} reports...')

print(f'\n[OK] Generated {len(hypotheses)} baseline reports')

# ── Compute metrics ────────────────────────────────────────────────────
print('\nComputing baseline metrics...')
# ✅ FIXED: Removed DEVICE parameter to accurately match your utils.metrics signature
results = compute_all_metrics(references, hypotheses)

# ── Save results ───────────────────────────────────────────────────────
os.makedirs('/kaggle/working/outputs', exist_ok=True)
with open('/kaggle/working/outputs/baseline_evaluation_results.txt', 'w') as f:
    f.write('BASELINE MODEL — EVALUATION RESULTS\n')
    f.write('=' * 40 + '\n\n')
    for k, v in results.items():
        f.write(f'  {k:<15} : {v}\n')

print('\n[OK] Baseline results saved to outputs/baseline_evaluation_results.txt')
print('\nNow compare both scores:')
print('Baseline  → outputs/baseline_evaluation_results.txt')
print('Full Model → outputs/evaluation_results.txt')

In [ ]:
%run /kaggle/working/evaluate_baseline.py

In [ ]:
import os
import zipfile

def zip_clean_code_only(output_filename='/kaggle/working/multimodal_cxr_code.zip'):
    source_dir = '/kaggle/working'
    
    with zipfile.ZipFile(output_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(source_dir):
            # Ignore binary folders and old zip copies
            if 'checkpoints' in root or 'outputs' in root:
                continue
            
            for file in files:
                if file.endswith('.zip') or file.endswith('.pt') or file.endswith('.pkl'):
                    continue
                    
                file_path = os.path.join(root, file)
                relative_path = os.path.relpath(file_path, source_dir)
                zipf.write(file_path, relative_path)
                
    print(f"[OK] Lightweight pure code layout zipped at: {output_filename}")
    print(f"New Downloadable Size: {os.path.getsize(output_filename) / 1024:.2f} KB")

zip_clean_code_only()

In [ ]:
import pandas as pd
import os

IMAGE_DIR = '/kaggle/input/datasets/raddar/chest-xrays-indiana-university/images/images_normalized'

df      = pd.read_csv('/kaggle/working/data/splits/test.csv')
samples = df.sample(5, random_state=42)

print('=' * 70)
print('GRADIO DEMO CHEAT SHEET')
print('Copy exactly what is shown below into the Gradio interface')
print('=' * 70)

for i, (_, row) in enumerate(samples.iterrows()):
    print(f'\n--- PATIENT {i+1} ---')
    print(f'Frontal path : {IMAGE_DIR}/{row["frontal_file"]}')
    print(f'Lateral path : {IMAGE_DIR}/{row["lateral_file"]}')
    print(f'Clinical note: {row["clinical_note"]}')
    print(f'Expected     : {row["report_text"][:80]}...')

In [ ]:
import os
import zipfile
from IPython.display import FileLink

# 1. Define the exact file paths for the 3 patients
image_paths = [
    # Patient 1
    "/kaggle/input/datasets/raddar/chest-xrays-indiana-university/images/images_normalized/2931_IM-1334-1001.dcm.png",
    "/kaggle/input/datasets/raddar/chest-xrays-indiana-university/images/images_normalized/2931_IM-1334-2001.dcm.png",
    # Patient 2
    "/kaggle/input/datasets/raddar/chest-xrays-indiana-university/images/images_normalized/3266_IM-1551-1001.dcm.png",
    "/kaggle/input/datasets/raddar/chest-xrays-indiana-university/images/images_normalized/3266_IM-1551-2001.dcm.png"
]

# Note: You didn't list paths for Patient 3, but you can add them to the list above if needed!

# 2. Define target destination zip file
zip_name = "/kaggle/working/patient_images.zip"

print("📦 Packing images into a zip file...")
with zipfile.ZipFile(zip_name, 'w') as img_zip:
    for path in image_paths:
        if os.path.exists(path):
            # Extract just the filename to save it cleanly inside the zip
            filename = os.path.basename(path)
            img_zip.write(path, arcname=filename)
            print(f"  ✅ Added: {filename}")
        else:
            print(f"  ❌ File not found: {path}")

print("\n🎉 Zip file created successfully!")
print("👇 Click the link below to download your images directly:")

# 3. Generate a clickable download link inside Kaggle
FileLink(zip_name)